<a href="https://colab.research.google.com/github/karsarobert/LLM2026/blob/main/PTE_GPT08.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gyakorló feladatok – Chat GPT és nagy nyelvi modellek
## OpenAI-kompatibilis API és eszközhívások (Tool Calling)

**PTE Gépi tanulás 8. gyakorlata**

---

## Feladatok áttekintése

| Feladat | Téma | Nehézség |
|---------|------|----------|
| 1. | Egyszerű API hívás | Alapszint |
| 2. | System prompt használata | Alapszint |
| 3. | JSON mód alkalmazása | Alapszint |
| 4. | Saját eszköz készítése | Középhaladó |
| 5. | Több eszköz együtt | Középhaladó |
| 6. | Komplex eszközrendszer | Haladó |

---

**FONTOS:** A feladatok előtt mindig állítsd be az API kapcsolatot!

## Beállítások – API kapcsolat létrehozása

Mielőtt bármelyik feladatot megoldanád, futtasd le ezt a cellát!
Ez hozza létre az OpenRouter-hez való kapcsolatot.

In [1]:
# ============================================================================
# 1. LÉPÉS: Az OpenAI csomag telepítése
# ============================================================================
# Az 'openai' csomagot telepítjük, amely lehetővé teszi az OpenAI-kompatibilis
# API-k elérését (OpenAI, OpenRouter, Groq, Ollama, stb.)

!pip install openai --quiet

# Ellenőrizzük, hogy sikeres volt-e a telepítés
print('✓ openai csomag sikeresen telepítve')

✓ openai csomag sikeresen telepítve


In [2]:
# ============================================================================
# 2. LÉPÉS: API kulcs betöltése
# ============================================================================
# Az API kulcsot betöltjük:
#   - Google Colab esetén: userdata.get() a titkos tárolóból
#   - Egyéb környezetben: getpass.getpass() interaktív bekérés

# FONTOS: Soha ne írd be közvetlenül a kulcsot a kódba!

try:
    # Google Colab titkos tárolójából próbáljuk betölteni
    from google.colab import userdata
    api_key = userdata.get('GROQ_API_KEY')
    print('✓ API kulcs betöltve (Colab titkos tárolóból)')
except Exception:
    # Ha nem Colab környezetben vagyunk, kéri a felhasználótól
    import getpass
    api_key = getpass.getpass('Add meg az OpenRouter API kulcsot: ')
    print('✓ API kulcs betöltve (manuálisan)')

✓ API kulcs betöltve (Colab titkos tárolóból)


In [3]:
# ============================================================================
# 3. LÉPÉS: OpenAI kliens létrehozása OpenRouter-re irányítva
# ============================================================================
# A kliens létrehozásánál a 'base_url' paraméterrel adjuk meg,
# hogy ne az OpenAI szerverére, hanem az OpenRouter-re küldjük a kéréseket.

# A különböző szolgáltatók URL-jei:
#   - OpenAI:        https://api.openai.com/v1
#   - OpenRouter:    https://openrouter.ai/api/v1
#   - Groq:          https://api.groq.com/openai/v1
#   - Ollama (helyi): http://localhost:11434/v1

from openai import OpenAI  # Az openai csomag fő osztálya

# Kliens létrehozása az OpenRouter végponttal
client = OpenAI(
    base_url='https://api.groq.com/openai/v1',  # OpenRouter API címe
    api_key=api_key                             # A mi titkos kulcsunk
)

# A használandó modell (ingyenes modell, ':free' végződés)
MODELL = 'openai/gpt-oss-120b'

print(f'✓ OpenAI kliens létrehozva')
print(f'✓ Modell: {MODELL}')

✓ OpenAI kliens létrehozva
✓ Modell: openai/gpt-oss-120b


In [4]:
# ============================================================================
# 4. LÉPÉS: Segédfüggvény a chat completion hívásokhoz
# ============================================================================
# Mivel a client.chat.completions.create() és valasz.choices[0].message.content
# elég hosszú, készítünk egy egyszerű segédfüggvényt a könnyebb használatért.

def kerdez(uzenet, system_prompt=None):
    """
    Egyszerű chat completion hívás.

    PARAMÉTEREK:
        uzenet (str): A felhasználó kérdése
        system_prompt (str, opcionalis): System prompt, ha szeretnénk irányítani az LLM viselkedését

    VISSZATÉRÉSI ÉRTÉK:
        str: Az LLM szöveges válasza
    """
    # Üzenetek listájának összeállítása
    messages = []

    # Ha van system prompt, azt hozzáadjuk az elejére
    # A system prompt láthatatlan utasítás a modellnek
    if system_prompt:
        messages.append({'role': 'system', 'content': system_prompt})

    # A felhasználó üzenete
    messages.append({'role': 'user', 'content': uzenet})

    # API hívás
    valasz = client.chat.completions.create(
        model=MODELL,
        messages=messages
    )

    # A válasz szövegének visszaadása
    return valasz.choices[0].message.content

print('✓ kerdez() segédfüggvény kész')

✓ kerdez() segédfüggvény kész


---

# 1. FELADAT: Egyszerű API hívás (Alapszint)

## Cél
Megérteni az OpenAI-kompatibilis API alap működését.

## Elméleti háttér
Az OpenAI API üzenet-alapú: egy `messages` listát küldünk, ahol minden üzenet
egy szótár `role` és `content` kulcsokkal.

```python
messages = [
    {"role": "user", "content": "Szia!"}
]
```

## Feladatok

1. Kérdezd meg az LLM-etől a saját neved és egy rövid köszönés kíséretében!
2. Írd ki a válasz struktúráját (típus, modell, szerepkör, tartalom)!
3. Kérdezd meg, mi a kedvenc számod 1-10 között, és add össze ezzel a számmal!

---

**Tipp:** A `kerdez()` függvény csak egy paramétert vár: a kérdés szövegét!

In [5]:
# ============================================================================
# 1.1 Kérdés feltevése – használd a kerdez() függvényt!
# ============================================================================
# Futtasd le ezt a cellát, és változtasd meg a kérdést tetszés szerint!
#
# PEPER: Cseréld ki a "Teszt Elek"-et a saját nevedre
#

valasz = kerdez('Szia! A nevem Teszt Elek. Kérlek köszönj nekem magyarul!')

print('LÁSD: Itt jelenik meg a válasz:')
print(valasz)

LÁSD: Itt jelenik meg a válasz:
Szia, Teszt Elek! Örülök, hogy írtál nekem. Remélem, szép napod van! 🌞


In [6]:
# ============================================================================
# 1.2 A válasz struktúrájának megvizsgálása
# ============================================================================
# Most megnézzük, milyen adatokat tartalmaz a válasz objektum.
# Ehhez NEM a kerdez() függvényt használjuk, hanem közvetlen API hívást,
# hogy elérjük a teljes válasz objektumot.

# API hívás – most a teljes válasz objektumot kapjuk vissza
valasz_objektum = client.chat.completions.create(
    model=MODELL,
    messages=[{'role': 'user', 'content': 'Szia! Ki vagy te?'}]
)

# A válasz struktúrájának részei:
print('=== A VÁLASZ STRUKTÚRÁJA ===')
print(f'Típus: {type(valasz_objektum)}')
print(f'Modell: {valasz_objektum.model}')
print(f'Válasz típusa: {type(valasz_objektum.choices[0].message)}')
print(f'Szerepkör: {valasz_objektum.choices[0].message.role}')
print(f'Tartalom: {valasz_objektum.choices[0].message.content}')
print(f'Token használat: {valasz_objektum.usage}')

=== A VÁLASZ STRUKTÚRÁJA ===
Típus: <class 'openai.types.chat.chat_completion.ChatCompletion'>
Modell: openai/gpt-oss-120b
Válasz típusa: <class 'openai.types.chat.chat_completion_message.ChatCompletionMessage'>
Szerepkör: assistant
Tartalom: Szia! Én a ChatGPT vagyok, egy mesterséges intelligencia, amelyet az OpenAI fejlesztett. Különböző témákban tudok segíteni: kérdések megválaszolása, információk keresése, szövegek írása vagy egyszerűen csak beszélgetni. Miben segíthetek Neked?
Token használat: CompletionUsage(completion_tokens=144, prompt_tokens=78, total_tokens=222, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=52, rejected_prediction_tokens=None), prompt_tokens_details=None, queue_time=0.004955825, prompt_time=0.003414642, completion_time=0.298994875, total_time=0.302409517)


In [7]:
# ============================================================================
# 1.3 Kérdés a kedvenc számmal (SAJÁT MEGOLDÁS)
# ============================================================================
# FELADAT: Kérdezd meg az LLM-et, mi a kedvenc száma 1-10 között,
# majd adj hozzá egy tetszőleges számot, és mond meg az összeget!
#
# PEPER: Írd meg a kódot ide!


# --- MEGOLDÁS (töröld ki a # jelet és futtasd) ---
# valasz = kerdez('Mi a kedvenc számod 1 és 10 között?')
# print(valasz)
# print()
# # Tipp: a kedvenc szám valószínűleg 7 lesz!
# osszeg = 7 + 5  # Példa: ha 7 a kedvenc szám, és mi hozzáadunk 5-öt
# print(f'Összeg: 7 + 5 = {osszeg}')

---

# 2. FELADAT: System prompt használata (Alapszint)

## Cél
Megérteni, hogyan befolyásolja a system prompt az LLM viselkedését.

## Elméleti háttér
Az OpenAI formátumban három szerepkör létezik:

| Szerepkör | Ki? | Mire jó? |
|-----------|-----|----------|
| `system` | A fejlesztő | Viselkedés, személyiség, szabályok |
| `user` | A felhasználó | Kérdések, utasítások |
| `assistant` | Az LLM | Korábbi válaszok (beszélgetéshez) |

A `system` prompt egy **láthatatlan utasítás** – a felhasználó nem látja, de az LLM követi.

## Feladatok

1. Kérdezd meg az LLM-etől, hogy adjon 3 tanácsot a Python tanulásához – **system prompt nélkül**!
2. Ugyanezt a kérdést – de most a system promptban mondd meg, hogy **rövid válaszokat** adjon!
3. Próbáld ki, hogy ugyanezt a kérdést tedd fel, de most legyen **időutazó történelmi személyiség**!

---

**Tipp:** A `kerdez()` második paramétere a system prompt!

In [8]:
# ============================================================================
# 2.1 System prompt NÉLKÜL – az LLM az alapértelmezett stílusában válaszol
# ============================================================================
# A kerdez() függvényt csak a kérdéssel hívjuk – nincs system prompt

print('=== SYSTEM PROMPT NÉLKÜL ===')
valasz_alap = kerdez('Adj 3 tanácsot a Python programozás tanulásához!')
print(valasz_alap)

=== SYSTEM PROMPT NÉLKÜL ===
**1. Kezdd a gyakorlati alappal!**  
   - **Interaktív tutorialok:** A *Codecademy*, a *freeCodeCamp* vagy a *Python.org* hivatalos „Python Tutorial” interaktív felületei lehetővé teszik, hogy kóddal kísérletezz anélkül, hogy előre fel kellene állítanod a környezetet.  
   - **Mini‑projektek:** Írj egyszerű programokat (pl. számkitaláló játék, szövegszámláló, CSV‑feldolgozó) már az első hétben – a “tanulás = csinálás” elv segít rögzíteni a szintaxisra vonatkozó ismereteket.

**2. Mélyedj el a “Pythonic” szemléletben!**  
   - **Olvasd a dokumentációt:** A *PEP 8* (stílusirányelvek) és a *The Zen of Python* (`import this`) megmutatja, hogyan írhatunk tiszta, olvasható kódot.  
   - **Használd a beépített adatstruktúrákat:** Listák, szótárak, halmazok és a list comprehension‑ek hatékonyabbá teszik a megoldásaidat, mint a manuális ciklusok. Gyakorold ezeket apró feladatokon (pl. “szűrd ki a páros számokat a listából list comprehension‑nel”).

**3. Csatlakozz a

In [9]:
# ============================================================================
# 2.2 System prompt – Rövid válaszok
# ============================================================================
# A system promptban megmondjuk az LLM-nek, hogyan viselkedjen.
# Ez a prompt láthatatlan a felhasználó számára, de az LLM követi!

# FONTOS: A system prompt a kerdez() második paramétere!

system_rövid = '''
Te egy segítőkész asszisztens vagy.
Válaszaid MINDIG rövidek legyenek: maximum 3 mondat.
Pontonként válaszolj, számozott listával.
'''

print('=== SYSTEM PROMPT: RÖVID VÁLASZOK ===')
valasz_rovid = kerdez(
    'Adj 3 tanácsot a Python programozás tanulásához!',
    system_prompt=system_rövid
)
print(valasz_rovid)

=== SYSTEM PROMPT: RÖVID VÁLASZOK ===
1. Gyakorolj rendszeresen kis feladatok megoldásával, például a LeetCode vagy a CodeSignal könnyű szintű kérdéseivel.  
2. Olvass jól strukturált forráskódot, és kövesd a PEP 8 stílusirányelveket a tiszta kód írásához.  
3. Használd a beépített dokumentációt (`help()`, `dir()`) és a Python hivatalos tutorialját, hogy megértsd az alapfogalmakat.


In [10]:
# ============================================================================
# 2.3 System prompt – Időutazó történelmi személyiség
# ============================================================================
# FONTOS: Próbálj ki egy kreatív system promptot!
# Az LLM-et rábírjuk, hogy egy történelmi személyiségként válaszoljon.

system_torténelmi = '''
Te vagy Szent-Györgyi Albert, a Nobel-díjas magyar biokémikus.
Ha kérdeznek, mindig utalj a saját élettapasztalataidra.
Válaszaid legyenek 2-3 mondatosak, és fejezd be úgy, hogy: ", ahogyan én mondanám."
'''

print('=== SYSTEM PROMPT: IDŐUTAZÓ TÖRTÉNELMI SZEMÉLYISÉG ===')
valasz_tortenelmi = kerdez(
    'Mi a véleményed a mai tudományos kutatásról?',
    system_prompt=system_torténelmi
)
print(valasz_tortenelmi)

=== SYSTEM PROMPT: IDŐUTAZÓ TÖRTÉNELMI SZEMÉLYISÉG ===
Úgy vélem, hogy a mai tudományos kutatás már sokat tanult a korábbi aszkorbinsav‑kutatásomból, de még mindig hiányzik az átfogó, integrált megközelítés. A kitartás és a kíváncsiság mindig is az én sikerem kulcsa volt, ahogyan én mondanám.


In [11]:
# ============================================================================
# 2.4 SAJÁT FELADAT – Próbáld ki a saját ötleteidet!
# ============================================================================
# PEPER: Írj egy saját system promptot és teszteld!
#
# Példák, amiket kipróbálhatsz:
#   - Te egy 8 éves gyerek vagy, aki mindent megmagyaráz
#   - Te egy szigorú tanár vagy, aki mindig kritizál
#   - Te egy humoros vígjáték író vagy
#   - Te egy filozófus vagy, aki mindent megkérdőjelez


# --- Írd ide a saját system promptodat! ---
# sajat_system = '''
#
# '''

# --- És ide a kérdésedet! ---
# sajat_valasz = kerdez('Szerinted mi a boldogság?', system_prompt=sajat_system)
# print(sajat_valasz)

---

# 3. FELADAT: JSON mód alkalmazása (Alapszint)

## Cél
Megtanulni, hogyan kérhetünk strukturált JSON kimenetet az LLM-től.

## Elméleti háttér
Az OpenAI-kompatibilis API-ban a `response_format={"type": "json_object"}`
paraméterrel kérhetjük a JSON kimenetet.

**Fontos:**
- A `response_format` paraméterrel kérjük a formátumot
- A **rendszer promptban** is meg kell mondani, hogy JSON-t várunk
- Az ingyenes modellek néha Markdown kódblokkba teszik a JSON-t
- Ezért szükség van egy `json_kinyerese()` függvényre!

## Feladatok

1. Kérj az LLM-től egy 3 fős névsort JSON formátumban (név, kor, foglalkozás)!
2. Kérdezd meg a kedvenc színeket szótár formátumban!
3. Készíts egy kedvencek listát (étel, szín, film) JSON tömbként!

---

**Tipp:** A JSON mód két dolgot igényel: `response_format` + system prompt!

In [12]:
# ============================================================================
# JSON kinyerő függvény (ismétlés a fő notebookból)
# ============================================================================
# Ez a függvény több módszert próbál a JSON kinyerésére:
#   1. Közvetlen json.loads() – ha tiszta JSON jött
#   2. Markdown kódblokk eltávolítása – ha ```json ... ``` van
#   3. Első { ... } keresés – ha extra szöveg van körülötte
#   4. Lista [ ... ] keresés – ha JSON tömb jött

import json

def json_kinyerese(szoveg):
    """Robusztus JSON kinyerés – kezeli a Markdown blokkot és extra szöveget."""
    szoveg = szoveg.strip()  # Eltávolítjuk a felesleges whitespace-t

    # 1. kísérlet: közvetlen parse – ha már tiszta JSON
    try:
        return json.loads(szoveg)
    except json.JSONDecodeError:
        pass  # Ha nem sikerült, próbáljuk a következő módszert

    # 2. kísérlet: Markdown kódblokk (```json ... ```) eltávolítása
    if '```json' in szoveg:
        e = szoveg.find('```json') + 7  # A '```json' után kezdődik
        v = szoveg.find('```', e)         # A záró ```
        if v != -1:
            try:
                return json.loads(szoveg[e:v].strip())
            except json.JSONDecodeError:
                pass

    # 3. kísérlet: első { ... utolsó } közötti rész
    e = szoveg.find('{')
    v = szoveg.rfind('}') + 1
    if e != -1 and v > e:
        try:
            return json.loads(szoveg[e:v])
        except json.JSONDecodeError:
            pass

    # 3b. kísérlet: lista [ ... ]
    e = szoveg.find('[')
    v = szoveg.rfind(']') + 1
    if e != -1 and v > e:
        try:
            return json.loads(szoveg[e:v])
        except json.JSONDecodeError:
            pass

    # Ha egyik sem sikerült, None-t adunk vissza
    return None

print('✓ json_kinyerese() függvény kész')

✓ json_kinyerese() függvény kész


In [13]:
# ============================================================================
# 3.1 JSON kimenet kérése – 3 fős névsor
# ============================================================================
# Két dolgot kell megadni:
#   1. response_format={'type': 'json_object'} – a kért formátum
#   2. system prompt – hogy biztosan JSON-t adjon

valasz = client.chat.completions.create(
    model=MODELL,

    # JSON mód bekapcsolása!
    response_format={'type': 'json_object'},

    messages=[
        # System prompt – EZ FONTOS!
        {'role': 'system', 'content': 'Mindig és kizárólag JSON formátumban válaszolj, semmilyen egyéb szöveget ne adj hozzá.'},
        # A kérdés
        {'role': 'user', 'content': 'Adj 3 ismert magyar tudós adatait! Mindegyiknek legyen: nev, szuletesiEv, foglalkozas.'}
    ]
)

# A nyers válasz (lehet, hogy Markdown blokkban van)
nyers_valasz = valasz.choices[0].message.content
print('=== NYERS VÁLASZ (első 200 karakter) ===')
print(nyers_valasz[:200])
print()

# A robusztus json_kinyerese() függvénnyel kinyerjük a JSON-t
eredmeny = json_kinyerese(nyers_valasz)

if eredmeny:
    print('=== KINYERT JSON (szépen formázva) ===')
    print(json.dumps(eredmeny, indent=2, ensure_ascii=False))
else:
    print('❌ HIBA: Nem sikerült JSON-t kinyerni!')

=== NYERS VÁLASZ (első 200 karakter) ===
[
{"nev":"Albert Szent-Györgyi","szuletesiEv":1893,"foglalkozas":"orvos, biokémikus"},{"nev":"John von Neumann","szuletesiEv":1903,"foglalkozas":"matematikus, informatikus"},{"nev":"Eötvös Loránd","sz

=== KINYERT JSON (szépen formázva) ===
[
  {
    "nev": "Albert Szent-Györgyi",
    "szuletesiEv": 1893,
    "foglalkozas": "orvos, biokémikus"
  },
  {
    "nev": "John von Neumann",
    "szuletesiEv": 1903,
    "foglalkozas": "matematikus, informatikus"
  },
  {
    "nev": "Eötvös Loránd",
    "szuletesiEv": 1848,
    "foglalkozas": "fizikus"
  }
]


In [14]:
# ============================================================================
# 3.2 JSON tömb kérése – Kedvencek listája
# ============================================================================
# Kérjünk egy listát a kedvenc ételeinkről, színeinkről, filmjeinkről!

valasz = client.chat.completions.create(
    model=MODELL,
    response_format={'type': 'json_object'},
    messages=[
        {'role': 'system', 'content': 'Mindig és kizárólag JSON formátumban válaszolj. Semmi más szöveg, csak a JSON.'},
        {'role': 'user', 'content': '''Készíts egy kedvencek listát tartalmazó JSON-t!
            Legyen benne:
            - etelek: tömb, benne 3 étel
            - szinek: tömb, benne 3 szín
            - filmek: tömb, benne 3 film'''}
    ]
)

eredmeny = json_kinyerese(valasz.choices[0].message.content)

if eredmeny:
    print('=== KEDVENCEK ===')

    # A JSON különböző típusú lehet – kezeljük mindkettőt!
    if isinstance(eredmeny, dict):
        print(f'Ételek: {eredmeny.get("etelek", [])}')
        print(f'Színek: {eredmeny.get("szinek", [])}')
        print(f'Filmek: {eredmeny.get("filmek", [])}')
    elif isinstance(eredmeny, list):
        for elem in eredmeny:
            print(f'  - {elem}')
else:
    print('❌ HIBA: Nem sikerült JSON-t kinyerni!')

=== KEDVENCEK ===
Ételek: ['Pizza', 'Sushi', 'Tészta']
Színek: ['Kék', 'Zöld', 'Piros']
Filmek: ['A nagy Gatsby', 'Matrix', 'Forrest Gump']


In [15]:
# ============================================================================
# 3.3 SAJÁT FELADAT – Próbálj ki különböző JSON struktúrákat!
# ============================================================================
# Írj egy kérdést, ami különböző JSON struktúrát kér!
#
# Példák:
#   - Könyvadatok: cim, szerzo, ev, oldalszam
#   - Autó adatok: marka, modell, evjarat, ar
#   - Zeneszerzők: nev, szuletes, halál, orszag


# --- Írd ide a saját kérdésedet! ---
# sajat_valasz = client.chat.completions.create(
#     model=MODELL,
#     response_format={'type': 'json_object'},
#     messages=[
#         {'role': 'system', 'content': 'Mindig és kizárólag JSON formátumban válaszolj.'},
#         {'role': 'user', 'content': '...ide írd a kérdésedet...'}
#     ]
# )
#
# eredmeny = json_kinyerese(sajat_valasz.choices[0].message.content)
# if eredmeny:
#     print(json.dumps(eredmeny, indent=2, ensure_ascii=False))

---

# 4. FELADAT: Saját eszköz készítése (Középhaladó)

## Cél
Megérteni, hogyan definiálhatunk és használhatunk saját eszközöket (tools).

## Elméleti háttér
Az LLM nem tud valós időben adatot lekérni. Az eszközhívás (Tool Calling)
lehetővé teszi, hogy az LLM "kérje" egy függvény meghívását.

**A folyamat:**
1. Felhasználó kérdez valamit
2. LLM: "Ehhez az X eszköz kell" → `tool_call`
3. A MI KÓDUNK végrehajtja a függvényt
4. Eredményt visszaküldjük az LLM-nek
5. LLM válaszol a felhasználónak

## Feladatok

1. Készíts egy `konyv_kereso(cim)` függvényt, ami visszaadja egy könyv adatait!
2. Definiáld az eszközt az API számára!
3. Integráld a `kerdez_eszkozzel()` rendszerbe!
4. Teszteld: "Keresd meg a 'romeo és julia' könyvet!"

---

**Tipp:** Az eszköz definíció egy JSON séma, ami az LLM számára írja le a függvényt!**

In [16]:
# ============================================================================
# 4.1 A szimulált könyvkereső függvény
# ============================================================================
# Ez a függvény szimulálja a könyvkereső API-t.
# Valós alkalmazásban itt lenne pl. egy Google Books API hívás.

def konyv_kereso(cim):
    """
    Szimulált könyvkereső API.

    PARAMÉTEREK:
        cim (str): A keresett könyv címe

    VISSZATÉRÉSI ÉRTÉK:
        str: JSON szöveg a könyv adataival
    """
    # Szimulált könyvadatbázis
    konyvek = {
        'romeo és julia': {
            'cim': 'Romeo és Júlia',
            'szerzo': 'William Shakespeare',
            'ev': 1597,
            'mufaj': 'Tragédia',
            'oldalszam': 180
        },
        'anna karenyina': {
            'cim': 'Anna Karenina',
            'szerzo': 'Lev Tolsztoj',
            'ev': 1877,
            'mufaj': 'Regény',
            'oldalszam': 864
        },
        'egri csillagok': {
            'cim': 'Egri csillagok',
            'szerzo': 'Gárdonyi Géza',
            'ev': 1901,
            'mufaj': 'Történelmi regény',
            'oldalszam': 456
        }
    }

    # Kisbetűsítjük a címet a kereséshez
    cim_kulcs = cim.lower().strip()

    if cim_kulcs in konyvek:
        return json.dumps(konyvek[cim_kulcs], ensure_ascii=False)
    else:
        return json.dumps({'hiba': f'A "{cim}" című könyv nem található'}, ensure_ascii=False)

# Teszteljük a függvényt!
print('=== KÖNYV KERESŐ – TESZT ===')
print(konyv_kereso('romeo és julia'))
print(konyv_kereso('ismeretlen könyv'))

=== KÖNYV KERESŐ – TESZT ===
{"cim": "Romeo és Júlia", "szerzo": "William Shakespeare", "ev": 1597, "mufaj": "Tragédia", "oldalszam": 180}
{"hiba": "A \"ismeretlen könyv\" című könyv nem található"}


In [17]:
# ============================================================================
# 4.2 Az eszköz definíciója az API számára
# ============================================================================
# Az eszköz definíció egy JSON séma – ez az LLM számára szóló "használati útmutató".
#
# FONTOS RESZEK:
#   - name: A függvény pontos neve
#   - description: Mit csinál a függvény (az LLM ezt érti meg!)
#   - parameters: Milyen paramétereket vár (típus, leírás)
#   - required: Melyik paraméterek kötelezőek

konyv_eszkoz = [
    {
        'type': 'function',
        'function': {
            # A függvény neve – ez kell a végrehajtáshoz!
            'name': 'konyv_kereso',

            # Leírás – ez az LLM-nek szól!
            # Legyen világos és pontos, hogy az LLM tudja, mikor hívja
            'description': 'Keres egy könyvet a cím alapján és visszaadja a könyv adatait (szerző, év, műfaj).',

            # Paraméterek sémája
            'parameters': {
                'type': 'object',
                'properties': {
                    'cim': {
                        'type': 'string',
                        'description': 'A keresett könyv címe (pl. "romeo és julia", "anna karenyina")'
                    }
                },
                'required': ['cim']  # A 'cim' paraméter kötelező!
            }
        }
    }
]

print('✓ Könyv eszköz definiálva!')
print(f'  Név: {konyv_eszkoz[0]["function"]["name"]}')
print(f'  Leírás: {konyv_eszkoz[0]["function"]["description"]}')

✓ Könyv eszköz definiálva!
  Név: konyv_kereso
  Leírás: Keres egy könyvet a cím alapján és visszaadja a könyv adatait (szerző, év, műfaj).


In [19]:
# ============================================================================
# 4.3 A teljes eszközhívási ciklus (ismétlés a fő notebookból)
# ============================================================================
# Ez a függvény automatikusan kezeli a teljes eszközhívási ciklust:
#   1. Kérdés küldése
#   2. Tool call végrehajtása (ha van)
#   3. Eredmény visszaküldése
#   4. Végső válasz kérése
def kerdez_eszkozzel(user_uzenet, eszkozok, fuggvenyek):
    """
    Teljes eszközhívási ciklus kezelése – CIKLUSOS verzió.

    Ez a verzió addig kérdez, amíg a modell már nem kér több eszközt.
    """
    # Üzenetek listájának kezdete
    uzenetek = [{'role': 'user', 'content': user_uzenet}]

    while True:  # ← CIKLUS: amíg van tool_call, addig folytatjuk
        # API hívás az eszközökkel
        valasz = client.chat.completions.create(
            model=MODELL,
            messages=uzenetek,
            tools=eszkozok,
            tool_choice="auto"
        )
        uzenet = valasz.choices[0].message

        # Ha nincs tool_call, kész vagyunk!
        if not uzenet.tool_calls:
            return uzenet.content

        # Tool call-ok végrehajtása
        uzenetek.append(uzenet)

        for tc in uzenet.tool_calls:
            fv_nev = tc.function.name
            fv_args = json.loads(tc.function.arguments)

            print(f'  🔧 Eszközhívás: {fv_nev}({fv_args})')

            # Üres kulcsok eltávolítása
            fv_args = {k: v for k, v in fv_args.items() if k}

            # A függvény végrehajtása
            eredmeny = fuggvenyek[fv_nev](**fv_args)
            print(f'  📤 Eredmény: {eredmeny}')

            # Eredmény visszaküldése
            uzenetek.append({
                'role': 'tool',
                'tool_call_id': tc.id,
                'content': eredmeny
            })
        # ← A ciklus visszaugrik és újra ellenőrzi, kell-e még tool

    # (Soha nem ide jutunk, de kell a while True-hoz)
    return "Hiba"



In [20]:
# ============================================================================
# 4.4 A könyv eszköz hozzáadása és tesztelés
# ============================================================================
# A könyv kereső függvényt hozzáadjuk a függvényszótárhoz
konyv_fuggvenyek = {
    'konyv_kereso': konyv_kereso
}

print('=== KÖNYV KERESŐ – ESZKÖZHÍVÁSSAL ===')
print()
valasz = kerdez_eszkozzel(
    'Keresd meg a "romeo és julia" könyvet és mond meg a szerzőjét!',
    konyv_eszkoz,
    konyv_fuggvenyek
)
print()
print('=== VÉGLEGES VÁLASZ ===')
print(valasz)

=== KÖNYV KERESŐ – ESZKÖZHÍVÁSSAL ===

  🔧 Eszközhívás: konyv_kereso({'cim': 'romeo és julia'})
  📤 Eredmény: {"cim": "Romeo és Júlia", "szerzo": "William Shakespeare", "ev": 1597, "mufaj": "Tragédia", "oldalszam": 180}

=== VÉGLEGES VÁLASZ ===
A keresett könyv szerzője: **William Shakespeare**.


In [21]:
# ============================================================================
# 4.5 SAJÁT FELADAT – Készíts egy másik eszközt!
# ============================================================================
# PEPER: Készíts egy új eszközt a következő ötletek egyikével!
#
# Javaslatok:
#   1. Zene kereső: zene_kereso(cim) -> szerző, album, év
#   2. Film kereső: film_kereso(cim) -> rendező, év, műfaj
#   3. Recept kereső: recept_kereso(nev) -> összetevők, elkészítési idő
#   4. Szó jelentése: szo_jelentese(szo) -> definíció, szófaj


# --- 1. LÉPÉS: Írd meg a függvényt ---
# def sajat_kereso(...):
#     ...
#     return json.dumps(..., ensure_ascii=False)


# --- 2. LÉPÉS: Definiáld az eszközt ---
# sajat_eszkoz = [
#     {
#         'type': 'function',
#         'function': {
#             'name': 'sajat_kereso',
#             'description': '...',
#             'parameters': {
#                 'type': 'object',
#                 'properties': {
#                     'param1': {
#                         'type': 'string',
#                         'description': '...'
#                     }
#                 },
#                 'required': ['param1']
#             }
#         }
#     }
# ]


# --- 3. LÉPÉS: Teszteld! ---
# sajat_fuggvenyek = {'sajat_kereso': sajat_kereso}
# valasz = kerdez_eszkozzel('...kérdés...', sajat_eszkoz, sajat_fuggvenyek)
# print(valasz)

---

# 5. FELADAT: Több eszköz együtt (Középhaladó)

## Cél
Több eszközt definiálni, és hagyni az LLM-et, hogy automatikusan válasszon köztük.

## Elméleti háttér
Az LLM egyszerre több eszközt is ismerhet – és a kérdés alapján
automatikusan kiválasztja a megfelelőt (vagy akár többet is).

## Feladatok

1. Készíts egy `diak_nyilvantartas(nev)` eszközt (hallgatói adatok)
2. Készíts egy `tantargy_lekerdez(tantargy)` eszközt (órarend)
3. Definiáld mindkét eszközt együtt!
4. Teszteld:
   - "Mik Anna jegyei?"
   - "Mik a mai órák?"
   - "Anna jegyei és a mai órarend?"

---

**Tipp:** Mindkét eszközt egy `eszkozok` listába tesszük, és az LLM dönti el, melyiket hívja!**

In [22]:
# ============================================================================
# 5.1 Két szimulált függvény
# ============================================================================

# --- Függvény 1: Diák nyilvántartás ---
def diak_nyilvantartas(nev):
    """
    Szimulált diák nyilvántartó rendszer.

    PARAMÉTEREK:
        nev (str): A diák teljes neve
    """
    diakok = {
        'Kovács Anna': {
            'nev': 'Kovács Anna',
            'evfolyam': 2,
            'jegyek': {'Matematika': 5, 'Fizika': 4, 'Programozás': 5}
        },
        'Szabó Péter': {
            'nev': 'Szabó Péter',
            'evfolyam': 1,
            'jegyek': {'Matematika': 3, 'Fizika': 3, 'Programozás': 4}
        },
        'Nagy Eszter': {
            'nev': 'Nagy Eszter',
            'evfolyam': 3,
            'jegyek': {'Matematika': 5, 'Fizika': 5, 'Programozás': 5}
        }
    }

    if nev in diakok:
        return json.dumps(diakok[nev], ensure_ascii=False)
    return json.dumps({'hiba': f'{nev} nem található a nyilvántartásban'}, ensure_ascii=False)


# --- Függvény 2: Tantárgy órarend ---
def tantargy_lekerdez(tantargy):
    """
    Szimulált órarend lekérdező.

    PARAMÉTEREK:
        tantargy (str): A tantárgy neve
    """
    orarend = {
        'Matematika': {
            'tantargy': 'Matematika',
            'oktato': 'Dr. Kovács János',
            'idopont': 'Hétfő 10:00-12:00',
            'helyszin': 'A101'
        },
        'Fizika': {
            'tantargy': 'Fizika',
            'oktato': 'Dr. Szabó Mária',
            'idopont': 'Szerda 14:00-16:00',
            'helyszin': 'B203'
        },
        'Programozás': {
            'tantargy': 'Programozás',
            'oktato': 'Dr. Tóth Péter',
            'idopont': 'Péntek 08:00-10:00',
            'helyszin': 'C305'
        }
    }

    if tantargy in orarend:
        return json.dumps(orarend[tantargy], ensure_ascii=False)
    return json.dumps({'hiba': f'{tantargy} tantárgy nincs az órarendben'}, ensure_ascii=False)

# Teszt
print('=== DIÁK TESZT ===')
print(diak_nyilvantartas('Kovács Anna'))
print()
print('=== TANTÁRGY TESZT ===')
print(tantargy_lekerdez('Matematika'))

=== DIÁK TESZT ===
{"nev": "Kovács Anna", "evfolyam": 2, "jegyek": {"Matematika": 5, "Fizika": 4, "Programozás": 5}}

=== TANTÁRGY TESZT ===
{"tantargy": "Matematika", "oktato": "Dr. Kovács János", "idopont": "Hétfő 10:00-12:00", "helyszin": "A101"}


In [23]:
# ============================================================================
# 5.2 Két eszköz együtt definiálása
# ============================================================================
# FONTOS: Mindkét eszközt egy listába tesszük!
# Az LLM automatikusan kiválasztja, melyiket hívja meg.

iskola_eszkozok = [
    {
        'type': 'function',
        'function': {
            'name': 'diak_nyilvantartas',
            'description': 'Lekérdezi egy diák adatait és jegyeit a név alapján.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'nev': {
                        'type': 'string',
                        'description': 'A diák teljes neve (pl. "Kovács Anna")'
                    }
                },
                'required': ['nev']
            }
        }
    },
    {
        'type': 'function',
        'function': {
            'name': 'tantargy_lekerdez',
            'description': 'Lekérdezi egy tantárgy adatait: oktató, időpont, helyszín.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'tantargy': {
                        'type': 'string',
                        'description': 'A tantárgy neve (pl. "Matematika", "Fizika")'
                    }
                },
                'required': ['tantargy']
            }
        }
    }
]

# A függvények szótára
iskola_fuggvenyek = {
    'diak_nyilvantartas': diak_nyilvantartas,
    'tantargy_lekerdez': tantargy_lekerdez
}

print('✓ 2 eszköz definiálva:', [e['function']['name'] for e in iskola_eszkozok])

✓ 2 eszköz definiálva: ['diak_nyilvantartas', 'tantargy_lekerdez']


In [24]:
# ============================================================================
# 5.3 Tesztelés: melyik eszközt hívja meg?
# ============================================================================

# --- Teszt 1: Csak diák adatok ---
print('=== 1. TESZT: DIÁK ADATOK ===')
valasz1 = kerdez_eszkozzel(
    'Mik Kovács Anna jegyei?',
    iskola_eszkozok,
    iskola_fuggvenyek
)
print(valasz1)
print()

=== 1. TESZT: DIÁK ADATOK ===
  🔧 Eszközhívás: diak_nyilvantartas({'nev': 'Kovács Anna'})
  📤 Eredmény: {"nev": "Kovács Anna", "evfolyam": 2, "jegyek": {"Matematika": 5, "Fizika": 4, "Programozás": 5}}
**Kovács Anna** tanuló jegyei (2. évfolyam):

| Tantárgy      | Jegy |
|--------------|------|
| Matematika   | 5 |
| Fizika       | 4 |
| Programozás  | 5 |

Ha további részletekre vagy más tantárgyakra van szükséged, szólj!



In [25]:
# --- Teszt 2: Csak tantárgy adatok ---
print('=== 2. TESZT: TANTÁRGY ADATOK ===')
valasz2 = kerdez_eszkozzel(
    'Mikor van a Fizika óra és ki az oktatója?',
    iskola_eszkozok,
    iskola_fuggvenyek
)
print(valasz2)
print()

=== 2. TESZT: TANTÁRGY ADATOK ===
  🔧 Eszközhívás: tantargy_lekerdez({'tantargy': 'Fizika'})
  📤 Eredmény: {"tantargy": "Fizika", "oktato": "Dr. Szabó Mária", "idopont": "Szerda 14:00-16:00", "helyszin": "B203"}
A Fizika órája:

- **Oktató:** Dr. Szabó Mária  
- **Időpont:** szerda 14:00‑16:00  
- **Helyszín:** B203  

Ha további információra van szükséged, nyugodtan kérdezz!



In [26]:
# --- Teszt 3: Mindkét eszközt egyszerre! ---
# Ez a legérdekesebb teszt: az LLM-nek kell döntenie!
print('=== 3. TESZT: MINDKÉT ESZKÖZ ===')
valasz3 = kerdez_eszkozzel(
    'Szabó Péter jegyei és a Programozás óra időpontja?',
    iskola_eszkozok,
    iskola_fuggvenyek
)
print(valasz3)

=== 3. TESZT: MINDKÉT ESZKÖZ ===
  🔧 Eszközhívás: diak_nyilvantartas({'nev': 'Szabó Péter'})
  📤 Eredmény: {"nev": "Szabó Péter", "evfolyam": 1, "jegyek": {"Matematika": 3, "Fizika": 3, "Programozás": 4}}
  🔧 Eszközhívás: tantargy_lekerdez({'tantargy': 'Programozás'})
  📤 Eredmény: {"tantargy": "Programozás", "oktato": "Dr. Tóth Péter", "idopont": "Péntek 08:00-10:00", "helyszin": "C305"}
**Szabó Péter jegyei**

| Tantárgy      | Jegy |
|--------------|------|
| Matematika   | 3 |
| Fizika       | 3 |
| Programozás  | **4** |

**Programozás óra információi**

- **Oktató:** Dr. Tóth Péter  
- **Időpont:** Péntek 08:00‑10:00  
- **Helyszín:** C305  

Ha további részletekre vagy más tantárgyakra van szükséged, szólj nyugodtan!


---

# 6. FELADAT: Komplex eszközrendszer (Haladó)

## Cél
Több, különböző típusú eszközt együtt használni egy komplex feladat megoldásához.

## A feladat
Készíts egy egyszerű "virtuális asszisztens" eszközkészletet!

**Készítsd el a következő 4 eszközt:**

| Eszköz | Függvény | Mire való |
|--------|----------|-----------|
| 1. Időjárás | `idojaras_lekerdez(varos)` | Város időjárása |
| 2. Valuta | `valuta_atvaltas(osszeg, forras, cel)` | Átváltás |
| 3. Naptár | `naptar_lekerdez()` | Mai nap adatai |
| 4. Ébresztő | `ebreszto_beallit(idopont, uzenet)` | Ébresztés |

**Kérdések, amiket tesztelned kell:**

1. "Milyen idő van Pécsett?"
2. "Mennyi 100 euró forintban?"
3. "Milyen nap van ma?"
4. "Állíts be ébresztőt holnapra 7 órára!"
5. "Budapesten milyen idő van és miben van 500 dollár?" (2 eszköz!)

---

**Tipp:** Minden eszközt regisztrálnod kell az `eszkozok` listában ÉS a `fuggvenyek` szótárban!**

In [27]:
# ============================================================================
# 6.1 Négy szimulált függvény
# ============================================================================

# --- Függvény 1: Időjárás ---
def idojaras_lekerdez(varos):
    """Szimulált időjárás API."""
    adatok = {
        'Pécs':     {'hofok': 18, 'allapot': 'napos', 'paratartalom': 45},
        'Budapest': {'hofok': 15, 'allapot': 'felhős', 'paratartalom': 60},
        'Debrecen': {'hofok': 12, 'allapot': 'esős', 'paratartalom': 80},
        'Szeged':   {'hofok': 20, 'allapot': 'napos', 'paratartalom': 40}
    }
    varos = varos.strip().capitalize()  # Egységesítés
    if varos in adatok:
        return json.dumps({'varos': varos, 'idojaras': adatok[varos]}, ensure_ascii=False)
    return json.dumps({'hiba': f'{varos} nem található'}, ensure_ascii=False)


# --- Függvény 2: Valuta átváltás ---
def valuta_atvaltas(osszeg, forras, cel):
    """Szimulált valutaátváltó."""
    arfolyamok = {
        ('EUR', 'HUF'): 395.0,
        ('HUF', 'EUR'): 0.0025,
        ('USD', 'HUF'): 370.0,
        ('HUF', 'USD'): 0.0027,
        ('EUR', 'USD'): 1.08,
        ('USD', 'EUR'): 0.93
    }
    forras = forras.upper()
    cel = cel.upper()
    kulcs = (forras, cel)

    if kulcs in arfolyamok:
        eredmeny = round(osszeg * arfolyamok[kulcs], 2)
        return json.dumps({
            'osszeg': osszeg,
            'forras': forras,
            'cel': cel,
            'eredmeny': eredmeny,
            'arfolyam': arfolyamok[kulcs]
        }, ensure_ascii=False)
    return json.dumps({'hiba': f'{forras} → {cel} átváltás nem elérhető'}, ensure_ascii=False)


# --- Függvény 3: Naptár ---
def naptar_lekerdez():
    """Szimulált naptár – mai nap adatai."""
    return json.dumps({
        'datum': '2026-03-16',
        'nap': 'hétfő',
        'nevnap': ['Henrietta'],
        'unnep': None
    }, ensure_ascii=False)

# --- Függvény 4: Ébresztő ---
def ebreszto_beallit(idopont, uzenet):
    """Szimulált ébresztő beállítás."""
    return json.dumps({
        'sikeres': True,
        'idopont': idopont,
        'uzenet': uzenet,
        'statusz': 'Ébresztő beállítva! 🔔'
    }, ensure_ascii=False)

# Teszt
print('=== ÖSSZES FÜGGVÉNY TESZT ===')
print('1. Időjárás:', idojaras_lekerdez('Pécs'))
print('2. Valuta:', valuta_atvaltas(100, 'EUR', 'HUF'))
print('3. Naptár:', naptar_lekerdez())
print('4. Ébresztő:', ebreszto_beallit('07:00', 'Kelj fel!'))

=== ÖSSZES FÜGGVÉNY TESZT ===
1. Időjárás: {"varos": "Pécs", "idojaras": {"hofok": 18, "allapot": "napos", "paratartalom": 45}}
2. Valuta: {"osszeg": 100, "forras": "EUR", "cel": "HUF", "eredmeny": 39500.0, "arfolyam": 395.0}
3. Naptár: {"datum": "2026-03-16", "nap": "hétfő", "nevnap": ["Henrietta"], "unnep": null}
4. Ébresztő: {"sikeres": true, "idopont": "07:00", "uzenet": "Kelj fel!", "statusz": "Ébresztő beállítva! 🔔"}


In [28]:
# ============================================================================
# 6.2 Mind a négy eszköz definiálása
# ============================================================================
# FONTOS: Minden eszközt hozzáadunk az 'eszkozok' listához!

asszisztens_eszkozok = [
    # --- 1. IDŐJÁRÁS ESZKÖZ ---
    {
        'type': 'function',
        'function': {
            'name': 'idojaras_lekerdez',
            'description': 'Lekérdezi egy magyar város aktuális időjárását (hőmérséklet, állapot, páratartalom).',
            'parameters': {
                'type': 'object',
                'properties': {
                    'varos': {
                        'type': 'string',
                        'description': 'A város neve (pl. Pécs, Budapest, Szeged)'
                    }
                },
                'required': ['varos']
            }
        }
    },

    # --- 2. VALUTA ÁTVÁLTÁS ESZKÖZ ---
    {
        'type': 'function',
        'function': {
            'name': 'valuta_atvaltas',
            'description': 'Átváltja az összeget egyik valutából a másikba. Támogatott valuták: HUF (forint), EUR (euró), USD (dollár).',
            'parameters': {
                'type': 'object',
                'properties': {
                    'osszeg': {
                        'type': 'number',
                        'description': 'Az átváltandó összeg'
                    },
                    'forras': {
                        'type': 'string',
                        'description': 'Forrás valuta kód (HUF, EUR, USD)'
                    },
                    'cel': {
                        'type': 'string',
                        'description': 'Cél valuta kód (HUF, EUR, USD)'
                    }
                },
                'required': ['osszeg', 'forras', 'cel']
            }
        }
    },

    # --- 3. NAPTÁR ESZKÖZ ---
    {
        'type': 'function',
        'function': {
            'name': 'naptar_lekerdez',
            'description': 'Visszaadja a mai nap adatait: dátum, hét napja, névnap.',
            'parameters': {
                'type': 'object',
                'properties': {},
                'required': []
            }
        }
    },

    # --- 4. ÉBRESZTŐ ESZKÖZ ---
    {
        'type': 'function',
        'function': {
            'name': 'ebreszto_beallit',
            'description': 'Beállít egy ébresztőt a megadott időpontra egy üzenettel.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'idopont': {
                        'type': 'string',
                        'description': 'Az ébresztő időpontja (pl. "07:00", "8:30")'
                    },
                    'uzenet': {
                        'type': 'string',
                        'description': 'Az ébresztő üzenete'
                    }
                },
                'required': ['idopont', 'uzenet']
            }
        }
    }
]

# A függvények szótára
asszisztens_fuggvenyek = {
    'idojaras_lekerdez': idojaras_lekerdez,
    'valuta_atvaltas': valuta_atvaltas,
    'naptar_lekerdez': naptar_lekerdez,
    'ebreszto_beallit': ebreszto_beallit
}

print(f'✓ {len(asszisztens_eszkozok)} eszköz definiálva!')
for e in asszisztens_eszkozok:
    print(f'   - {e["function"]["name"]}')

✓ 4 eszköz definiálva!
   - idojaras_lekerdez
   - valuta_atvaltas
   - naptar_lekerdez
   - ebreszto_beallit


In [29]:
# ============================================================================
# 6.3 Tesztelés – minden eszköz külön-külön és együtt
# ============================================================================

print('=' * 60)
print('1. TESZT: IDŐJÁRÁS')
print('=' * 60)
valasz = kerdez_eszkozzel(
    'Milyen idő van most Pécsett?',
    asszisztens_eszkozok,
    asszisztens_fuggvenyek
)
print(valasz)

1. TESZT: IDŐJÁRÁS
  🔧 Eszközhívás: idojaras_lekerdez({'varos': 'Pécs'})
  📤 Eredmény: {"varos": "Pécs", "idojaras": {"hofok": 18, "allapot": "napos", "paratartalom": 45}}
Jelenleg Pécsett **18 °C**, napsütéses idő és a páratartalom körülbelül **45 %**.


In [30]:
print('=' * 60)
print('2. TESZT: VALUTA ÁTVÁLTÁS')
print('=' * 60)
valasz = kerdez_eszkozzel(
    'Mennyi 100 euró forintban?',
    asszisztens_eszkozok,
    asszisztens_fuggvenyek
)
print(valasz)

2. TESZT: VALUTA ÁTVÁLTÁS
  🔧 Eszközhívás: valuta_atvaltas({'cel': 'HUF', 'forras': 'EUR', 'osszeg': 100})
  📤 Eredmény: {"osszeg": 100, "forras": "EUR", "cel": "HUF", "eredmeny": 39500.0, "arfolyam": 395.0}
100 EUR ≈ 39 500 HUF (az aktuális árfolyam ≈ 395 HUF/EUR).


In [31]:
print('=' * 60)
print('3. TESZT: NAPTÁR')
print('=' * 60)
valasz = kerdez_eszkozzel(
    'Milyen nap van ma? Kinek van névnapja?',
    asszisztens_eszkozok,
    asszisztens_fuggvenyek
)
print(valasz)

3. TESZT: NAPTÁR
  🔧 Eszközhívás: naptar_lekerdez({})
  📤 Eredmény: {"datum": "2026-03-16", "nap": "hétfő", "nevnap": ["Henrietta"], "unnep": null}
Ma 2026. március 16. van, hétfő. A névnapra **Henrietta** név jut ma.


In [32]:
print('=' * 60)
print('4. TESZT: ÉBRESZTŐ')
print('=' * 60)
valasz = kerdez_eszkozzel(
    'Állíts be ébresztőt holnapra 7 órára!',
    asszisztens_eszkozok,
    asszisztens_fuggvenyek
)
print(valasz)

4. TESZT: ÉBRESZTŐ
  🔧 Eszközhívás: ebreszto_beallit({'idopont': '07:00', 'uzenet': 'Ébresztő holnap reggel 7 órára'})
  📤 Eredmény: {"sikeres": true, "idopont": "07:00", "uzenet": "Ébresztő holnap reggel 7 órára", "statusz": "Ébresztő beállítva! 🔔"}
Az ébresztő beállítva: holnap reggel 07:00-kor fog megszólalni a megadott üzenettel. 🔔

Ha bármi mást is szeretnél, csak szólj!


In [33]:
print('=' * 60)
print('5. TESZT: KÉT ESZKÖZ EGYSZERRE (időjárás + valuta)')
print('=' * 60)
valasz = kerdez_eszkozzel(
    'Budapesten milyen idő van? Mennyi 500 dollár forintban?',
    asszisztens_eszkozok,
    asszisztens_fuggvenyek
)
print(valasz)

5. TESZT: KÉT ESZKÖZ EGYSZERRE (időjárás + valuta)
  🔧 Eszközhívás: idojaras_lekerdez({'varos': 'Budapest'})
  📤 Eredmény: {"varos": "Budapest", "idojaras": {"hofok": 15, "allapot": "felhős", "paratartalom": 60}}
  🔧 Eszközhívás: valuta_atvaltas({'cel': 'HUF', 'forras': 'USD', 'osszeg': 500})
  📤 Eredmény: {"osszeg": 500, "forras": "USD", "cel": "HUF", "eredmeny": 185000.0, "arfolyam": 370.0}
Budapesten jelenleg **15 °C**, felhős az időjárás, a páratartalom pedig körülbelül **60 %**.

Az 500 USD átváltva forintba **185 000 HUF** (az átváltási árfolyam körülbelül 370 HUF/USD).


---

## Összefoglalás

Ebben a gyakorló notebookban a következőket gyakoroltuk:

| Feladat | Téma | Mit tanultál? |
|---------|------|---------------|
| 1. | Egyszerű API hívás | Hogyan működik a chat completion |
| 2. | System prompt | Hogyan irányítjuk az LLM viselkedését |
| 3. | JSON mód | Strukturált adatok kérése |
| 4. | Saját eszköz | Eszköz definíció és használat |
| 5. | Több eszköz | Az LLM automatikus eszközválasztása |
| 6. | Komplex rendszer | Valós idejű asszisztens építése |

**Következő lépések:**

1. Próbálj ki saját eszközöket a Feladat 4.5 és 5.4 részekben!
2. Kombinálj több eszközt egyetlen kérdésben!
3. Írj system promptot, ami "személyiséget" ad az asszisztensednek!

---